In [56]:
from core.config import ConfigManager
from astropy.coordinates import SkyCoord
from pathlib import Path
import os
from core.logger import PipelineLogger
from core.directory_manager import DirectoryManager
from core.hdf5_handler import HDF5Handler
from core.map_tools import MapGenerator

In [57]:
config = ConfigManager('config.yaml')
method = config.get('fitting_procedure')
coordsys = config.get('coordinates.coord_sys', 'equatorial')
output_path = config.get('fitting.output_dir')
output_dir_name = config.get('fitting.fit_name')

# Test logger
logger = PipelineLogger('./logs')
logger.info("Pipeline test")

directory_manager = DirectoryManager(output_path, output_dir_name, logger=logger)
directory_manager.create_structure()

ra = config.get('coordinates.ra')
dec = config.get('coordinates.dec')
if ra is None or dec is None:
    l = config.get('coordinates.l')
    b = config.get('coordinates.b')
    if l is None or b is None:
        raise ValueError("Config must set either (ra, dec) or (l, b) for the ROI center")
    c = SkyCoord(l, b, frame='galactic', unit='deg')
    ra = float(c.icrs.ra.deg)
    dec = float(c.icrs.dec.deg)
    logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
    config.set('coordinates.ra', ra)
    config.set('coordinates.dec', dec)


print(f"Fitting method: {method}")
print(f"Coordinate system: {coordsys}")
print(f"RA: {ra}")
print(f"Dec: {dec}")
print(f"Output path: {output_path}")
print(f"Output directory name: {output_dir_name}")

2026-08-20 20:22:50 - Pipeline - INFO - Pipeline logger initialized
2026-08-20 20:22:50 - Pipeline - INFO - Log level: INFO
2026-08-20 20:22:50 - Pipeline - INFO - Pipeline log: logs/pipeline_20260820_202250.log
2026-08-20 20:22:50 - Pipeline - INFO - Full log: logs/full_log_20260820_202250.log
2026-08-20 20:22:50 - Pipeline - INFO - Pipeline test
2026-08-20 20:22:50 - Pipeline - INFO - Initialized DirectoryManager at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1
2026-08-20 20:22:50 - Pipeline - INFO - Directory structure created at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1
2026-08-20 20:22:50 - Pipeline - INFO - Converted galactic coordinates (l=12.5, b=0) to equatorial (RA=273.2187758272126, Dec=-18.100800822935284)


Fitting method: Drips
Coordinate system: G
RA: 273.2187758272126
Dec: -18.100800822935284
Output path: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/
Output directory name: Run1


In [58]:
###Drips 
step_dir = directory_manager.get_step_results_dir('Step0-Allpoint-sources')

In [ ]:
def _build_significance_map():
        """Build the significance map from count maps if coordinates.create_sig_map."""
        if not config.get('coordinates.create_sig_map', False):
            return None
        sig_map_path_cfe = config.get('coordinates.sig_map_path')
        if sig_map_path_cfe:
            sig_map_path = Path(sig_map_path_cfe)
        else:
            sig_map_path = directory_manager.get_datamap_dir() / "significance_map.fits"
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            config.set("coordinates.sig_map_path", str(sig_map_path))
        if sig_map_path.exists():
            logger.info(f"Significance map already exists at {sig_map_path}, skipping generation")
            return sig_map_path
        
        # checkpoint.save_step('build_significance_map', 0, 'running', {})
        count_map_dir = config.get('coordinates.count_map_dir')
        image_bins = config.get('coordinates.image_bins')
        detector_response = config.get('coordinates.detector_response')

        fits_mapping = MapGenerator.find_fits_files_by_bins(count_map_dir, image_bins, logger=logger)
        if not fits_mapping:
            checkpoint.save_step('build_significance_map', 0, 'failed', {'error': 'no count-map FITS files found'})
            raise RuntimeError(f"No count-map FITS files found in {count_map_dir} for bins {image_bins}")
        ra = config.get('coordinates.ra')
        dec = config.get('coordinates.dec')
        if ra is None or dec is None:
            l = config.get('coordinates.l')
            b = config.get('coordinates.b')
            skycoord = SkyCoord(l, b, frame='galactic', unit='deg')
            ra = skycoord.icrs.ra.deg
            dec = skycoord.icrs.dec.deg
            logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
            config.set('coordinates.ra', ra)
            config.set('coordinates.dec', dec) 
        
        output_path = MapGenerator.create_healpix_map(
            input_fits_files=list(fits_mapping.values()),
            energy_bins=list(fits_mapping.keys()),
            detector_response=detector_response,
            ra_center=float(config.get('coordinates.ra')),
            dec_center=float(config.get('coordinates.dec')),
            roi_x=float(10.0),
            roi_y=float(10.0),
            output_file=str(sig_map_path),
            logger=logger,
            pixi_manifest_path=config.get('alps.pixi_aerie_folder'),
        )


In [60]:
_build_significance_map()

2026-08-20 20:22:54 - Pipeline - INFO - Significance map already exists at /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDir/significance_map.fits, skipping generation
2026-08-20 20:22:54 - Pipeline - INFO - Searching for FITS files in /Users/rishi/Documents/Analysis/data/fhitcountmaps/
2026-08-20 20:22:54 - Pipeline - INFO - Looking for bins: ['B5C0', 'B6C0', 'B7C0', 'B8C0', 'B9C0', 'B10C0']
2026-08-20 20:22:54 - Pipeline - INFO - Found 6/6 energy bins
2026-08-20 20:22:54 - Pipeline - INFO - Creating HEALPix map from 6 FITS files
2026-08-20 20:22:54 - Pipeline - INFO - Energy bins: ['B5C0', 'B6C0', 'B7C0', 'B8C0', 'B9C0', 'B10C0']
2026-08-20 20:22:54 - Pipeline - INFO - Executing HealpixSigFluxMap command:
2026-08-20 20:23:20 - Pipeline - INFO - HEALPix map created successfully: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDir/signi

In [61]:
%load_ext autoreload
%autoreload 2
from drips_seeder import DRIPSSeeder
config = ConfigManager('config.yaml')
seeder = DRIPSSeeder(config, logger, directory_manager, step_path=str(step_dir))
drip_model_path = seeder.run()

2026-08-20 20:23:20 - Pipeline - INFO - Using significance map created: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDir/significance_map.fits
2026-08-20 20:23:20 - Pipeline - INFO - Using galactic coordinates from config: (l=12.5, b=0.0)
2026-08-20 20:23:20 - Pipeline - INFO - ROI size: 3.0° x 5.0°
2026-08-20 20:23:20 - Pipeline - INFO - Output directory for seeding: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources
2026-08-20 20:23:20 - Pipeline - INFO - Running seed model search in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources
2026-08-20 20:23:20 - Pipeline - INFO - Loading HAWC data from /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/DataMapDi

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ROI center in Galactic Coordintes = 12.5, 0.0
Loading Galactic Map
Fits File loaded
Degrees per pixel: 0.005555555555555556 
  Map shape  : 1800 × 1080  |  pixel size: 0.0056°
  Sig range  : [-1637500000000000486439473119232.00, 31.10]
Peak intensity pixel location: (np.int64(905), np.int64(469))
Peak intensity sky location: <SkyCoord (Galactic): (l, b) in deg
    (12.88888912, 0.03333329)>
Peak intensity value: 31.103438189014234
Plotting contours [7, 9, 12, 13, 14, 15]
  Max significance 31.10sigma exceeds threshold 5.0sigma — proceeding with analysis.
  Image softly floored to -5sigma


Processing radius 0.25°:   0%|          | 0/4 [00:00<?, ?it/s]

Number of pixels corresponding to 0.25 smear radius = 45.00
Estimated background RMS: 0.02561798303482701


Processing radius 0.30°:  25%|██▌       | 1/4 [00:05<00:16,  5.65s/it]

...

Raw blobs — point source: 5, extended: 2
Sources after 5$\sigma$ filtering: 4
Number of pixels corresponding to 0.50 smear radius = 90.00
Estimated background RMS: 0.03619823039236246


Raw blobs — point source: 6, extended: 1
Sources after 5$\sigma$ filtering: 4
  PS  after all cuts:    3
  EXT after all cuts:    2
Plotting contours [7, 9, 12, 13, 14, 15]


Intensity Fraction of pixels greater than 5 sigma detection threshold = 97.8%
Larger blob coord = (np.float64(13.544449123938188), np.float64(0.09444347439460736))
  No smaller blobs overlapping larger blob — tagging as EXT
  PS blob at (x=327, y=974, r=27.00 pixels) with sep=0.349° and no overlap but high TS difference (-0.4) — tagging as PS
PS flagged: 0
Intensity Fraction of pixels greater than 5 sigma detection threshold = 76.5%
Larger blob coord = (np.float64(10.79442831644101), np.float64(0.044443214836184644))
  No smaller blobs overlapping larger blob — tagging as EXT
  PS blob at (x=797, y=918, r=27.00 pixels) with sep=0.279° and no overlap but high TS difference (-8.2) — tagging as PS
PS flagged: 0
  Kept   — PS: 1  EXT: 2
  Removed— PS: 2  EXT: 0
Final — PS kept: 1  EXT kept: 2
        PS removed: 2  EXT removed: 0
  Kept   — PS: 1  EXT: 2
  Removed— PS: 2  EXT: 0


2026-08-20 20:23:46 - Pipeline - INFO - Found 3 sources, 3 within original ROI
2026-08-20 20:23:46 - Pipeline - INFO - Results saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/filtered_sources.yaml
2026-08-20 20:23:46 - Pipeline - INFO - Results saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/filtered_sources.yaml


Converting blob at (x=351.0, y=916.0, r=90.00 pixels) to coord (RA=273.656°, Dec=-17.139°) with radius 0.50°
Converting blob at (x=846.0, y=907.0, r=90.00 pixels) to coord (RA=272.309°, Dec=-19.574°) with radius 0.50°
Converting blob at (x=473.0, y=902.0, r=27.00 pixels) to coord (RA=273.388°, Dec=-17.771°) with radius 0.15°
Plotting contours [7, 9, 12, 13, 14, 15]
Model saved to: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model


In [62]:
from __future__ import division

import hashlib
from astropy import wcs
import astropy.units as u
import numpy as np
from astropy.coordinates import ICRS, BaseCoordinateFrame, SkyCoord, Angle
from astropy.io import fits
from past.utils import old_div
from scipy.interpolate import RegularGridInterpolator
import scipy
from astromodels.functions.function import Function3D, FunctionMeta
from astromodels.utils.angular_distance import angular_distance_fast
from astropy.coordinates.angles.utils import angular_separation, position_angle
from astromodels.utils.angular_distance import angular_distance

#############Created from Hugo's Galprop Template Analysis#########

class Hermes(Function3D, metaclass=FunctionMeta):
    r"""
        description :

            Use a 3D template that has morphology and flux information.
            GalProp, DRAGON or a similar model in fits format would work. 
            Only parameter is a normalization factor. 

        latex : $ N $

        parameters :

            N :

                desc : normalization
                initial value : 1
                fix : yes

            hash :

                desc : hash of model map [needed for memoization]
                initial value : 1
                fix : yes

            ihdu:
                desc: header unit index of fits file
                initial value: 0
                fix: True
                min: 0

        properties:
            fits_file:
                desc: fits file to load
                defer: True
                function: _load_file
            frame:
                desc: coordinate frame
                initial value: icrs
                allowed values:
                    - icrs
                    - galactic
                    - fk5
                    - fk4
                    - fk4_no_e
    """

    def _set_units(self, x_unit, y_unit, z_unit, w_unit):
        #self.N.unit = (u.keV * u.cm**2 * u.s * u.sr) ** (-1)
        # The spectrum and morphology are embedded in the template.
        # The normalization N is for scaling up or down the whole template to fit the data.
        self.N.unit = w_unit

    def _setup(self):
        self._frame = "icrs"
        self._intmap = None

    def set_frame(self, new_frame):
        """
        Set a new frame for the coordinates (the default is ICRS J2000)

        :param new_frame: a coordinate frame from astropy
        :return: (none)
        """
        assert isinstance(new_frame, BaseCoordinateFrame)

        self._frame = new_frame

    def _load_file(self):

        if self.fits_file is None:
            raise RuntimeError(
                "Need to specify a fits file with a template map.")

        self._fitsfile=self.fits_file.value

        with fits.open(self._fitsfile) as f:

            self._delLon = f[int(self.ihdu.value)].header['CDELT1']
            self._delLat = f[int(self.ihdu.value)].header['CDELT2']
            self._delEn = f[int(self.ihdu.value)].header['CDELT3']
            self._refLon = f[int(self.ihdu.value)].header['CRVAL1']
            self._refLat = f[int(self.ihdu.value)].header['CRVAL2']
            self._refEn = f[int(self.ihdu.value)].header['CRVAL3']  # values in log10
            self._map = f[int(self.ihdu.value)].data
            self._wcs = wcs.WCS(header = f[int(self.ihdu.value)].header)
            if len(self._map.shape) == 4:
                self._map = self._map[0]
            self._nl = f[int(self.ihdu.value)].header['NAXIS1']  # longitude
            self._nb = f[int(self.ihdu.value)].header['NAXIS2']  # latitude
            self._ne = f[int(self.ihdu.value)].header['NAXIS3']  # energy

            # Create the function for the interpolation
            self._L = np.linspace(
                self._refLon, self._refLon-(self._nl-1)*self._delLon, self._nl)
            self._B = np.linspace(
                self._refLat, self._refLat+(self._nb-1)*self._delLat, self._nb)
            self._E = np.linspace(
                self._refEn, self._refEn+(self._ne-1)*self._delEn, self._ne)
            for i in range(len(self._E)):
                self._map[i] = np.fliplr(self._map[i])
            self._F = RegularGridInterpolator(
                (self._E, self._B, self._L), self._map, bounds_error=False)

            h = hashlib.sha224()
            h.update(self._map)
            h.update(repr(self._wcs).encode("utf-8"))
            self.hash = int(h.hexdigest(), 16)


    def evaluate(self, x, y, z, N, hash, ihdu):

        if self._map is None:
            self._load_file(self._fitsfile)

        if self._intmap is None:
            _coord = SkyCoord(ra=x, dec=y, frame=self._frame, unit="deg")
            b = _coord.transform_to('galactic').b.value
            l = _coord.transform_to('galactic').l.value
            lon = l
            lat = b
            energy = np.log10(z)

            if lon.size != lat.size:
                raise AttributeError("Lon and Lat should be the same size")
            f = np.zeros([lon.size, energy.size])
            E0 = self._refEn
            Ef = self._refEn + (self._ne-1)*self._delEn

            #shift = np.where(lon > 180.)
            #lon[shift] = 180 - lon[shift]

            for i in range(energy.size):
                e = np.repeat(energy[i], len(lon))
                f[:, i] = self._F(np.array([e, lat, lon]).T)

            bad_idx = np.isnan(f)
            f[bad_idx] = 0
            bad_idx = np.isinf(f)
            f[bad_idx] = 0
            bad_idx = np.isneginf(f)
            f[bad_idx] = 0
            assert np.all(np.isfinite(f)), "some interpolated values are wrong"
            self._intmap = f

        A = np.multiply(N, self._intmap)
        return A

    def get_boundaries(self):
        # Taken from SpatialTemplate_2D
        Xcorners = np.array( [0, 0,        self._nl, self._nl] )
        Ycorners = np.array( [0, self._nb, 0,        self._nb] )

        corners = SkyCoord.from_pixel( Xcorners, Ycorners, wcs=self._wcs, origin = 0).transform_to(self._frame)

        min_lon = min(corners.ra.degree)
        max_lon = max(corners.ra.degree)

        min_lat = min(corners.dec.degree)
        max_lat = max(corners.dec.degree)

        return (min_lon, max_lon), (min_lat, max_lat)


    def get_total_spatial_integral(self, z=None):
        if isinstance( z, u.Quantity):
            z = z.value
        return np.multiply(self.N.value, np.ones_like( z ))



In [65]:
%load_ext autoreload
%autoreload 2
import importlib
import source_fitter
importlib.reload(source_fitter)
logger.info('Starting source_fitter (DRIPS-seeded in-process fit)')

fit_output = source_fitter.run_joint_fit(drip_model_path, config, logger, directory_manager)

2026-08-20 20:24:38 - Pipeline - INFO - Starting source_fitter (DRIPS-seeded in-process fit)
2026-08-20 20:24:38 - Pipeline - INFO - Running joint fit on DRIPS seed model (/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model) in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit
2026-08-20 20:24:38 - Pipeline - INFO - Fitting model /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step0-Allpoint-sources/curModel.model in /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit  with compute_err=False, compute_TS=False, make_maps=True
2026-08-20 20:24:38 - Pipeline - INFO - Map tree: /Users/rishi/Documents/Analysis/data/maptree-fhit2pct-pas

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-08-20 20:24:38 - Pipeline - INFO - No ROI template provided, computing ROI from provided ra and dec
2026-08-20 20:24:38 - Pipeline - INFO - ROI center: 273.2187758272126, -18.100800822935284
2026-08-20 20:24:38 - Pipeline - INFO - Running threeML fit pipeline with bins: ['B5C0', 'B6C0', 'B7C0', 'B8C0', 'B9C0', 'B10C0']


['Source0', 'Source1', 'Source2']
273.2187758272126 -18.100800822935284 15
Center of ROI  RA: 273.2188 Dec: -18.1008 Radius: 15.00
Source         RA:          Dec:           Distance to center
Source0          273.3881    -17.7710   0.367 degrees
Source1          273.6564    -17.1386   1.049 degrees
Source2          272.3091    -19.5738   1.706 degrees


2026-08-20 20:24:53 - Pipeline - INFO - Running MLE without error estimation


...

Values of -log(likelihood) at the minimum:

,-log(likelihood)
HAWC,117511.160316
total,117511.160316


Values of statistical measures:

,statistical measures
AIC,235052.320860
BIC,235240.704563


2026-08-20 20:25:26 - Pipeline - INFO - Fit results saved to /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/likelihoodResults.fits
2026-08-20 20:25:26 - Pipeline - INFO - Saving HAL output maps
2026-08-20 20:25:26 - Pipeline - INFO - SAVE A BIG MAP
2026-08-20 20:25:49 - Pipeline - INFO - Writing model map...
2026-08-20 20:25:50 - Pipeline - INFO - Fit Step1-JointFit: -logL=117511.160, AIC=235052.321 (1.20 min)
20:24:42 WARNING   The naima package is not available. Models    ]8;id=3235262;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=3235263;file:///Users/rishi/Documents/Analysis/aerie/.pixi/envs/threeml/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  that depend on it will not be available                       
2

In [66]:
def _build_fit_maps(config, logger, directory_manager, path, name, checkpoint=None):
    """Create significance maps."""

    ra = config.get('coordinates.ra')
    dec = config.get('coordinates.dec')
    if ra is None or dec is None:
        l = config.get('coordinates.l')
        b = config.get('coordinates.b')
        skycoord = SkyCoord(l, b, frame='galactic', unit='deg')
        ra = skycoord.icrs.ra.deg
        dec = skycoord.icrs.dec.deg
        logger.info(f"Converted galactic coordinates (l={l}, b={b}) to equatorial (RA={ra}, Dec={dec})")
        config.set('coordinates.ra', ra)
        config.set('coordinates.dec', dec) 
    
    
    created_files = HDF5Handler.convert_hd5_to_fits(
        input_dir=str(path),
        hd5_filename='residual_fit.hd5',
        output_basename='residual',
        logger=logger
    )
    print(f"Created FITS files: {created_files}")
    bins = config.get('fitting.bins')
    detector_response = config.get('coordinates.detector_response')
    print(f"Path: {path}")

    if os.path.exists(str(path / 'fits' / f'{name}.fits')):
        logger.info(f"Output FITS file {path / 'fits' / f'{name}.fits'} already exists, skipping map generation")
        return path / 'fits' / f'{name}.fits' 
    output_path = MapGenerator.create_healpix_map(
        input_fits_files=list(created_files),
        energy_bins=list(bins),
        detector_response=detector_response,
        ra_center=float(config.get('coordinates.ra')),
        dec_center=float(config.get('coordinates.dec')),
        roi_x=float(config.get('coordinates.roi_x', 4.0)*2.5),
        roi_y=float(config.get('coordinates.roi_y', 4.0)*2),
        output_file=str(path / 'fits' / f'{name}.fits'),
        logger=logger,
        pixi_manifest_path=config.get('alps.pixi_aerie_folder'),
    )

    return output_path
# def check_hotspots(residual_map_path):


In [67]:
resmap = _build_fit_maps(config, logger, directory_manager, fit_output.step_dir, 'residual')

2026-08-20 20:25:50 - Pipeline - INFO - Converted galactic coordinates (l=12.5, b=0) to equatorial (RA=273.2187758272126, Dec=-18.100800822935284)
2026-08-20 20:25:50 - Pipeline - INFO - Converting HDF5 to FITS: residual_fit.hd5


2026-08-20 20:25:54 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB5C0.fits.gz
2026-08-20 20:25:57 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB6C0.fits.gz
2026-08-20 20:26:00 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB7C0.fits.gz
2026-08-20 20:26:02 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB8C0.fits.gz
2026-08-20 20:26:04 - Pipeline - INFO - Created FITS file: /Users/rishi/Documents/Analys

Created FITS files: [PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB5C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB6C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB7C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB8C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual_binB9C0.fits.gz'), PosixPath('/Users/rishi/Documents/Analysis/Sources/AstroImageDet

2026-08-20 20:26:28 - Pipeline - INFO - HEALPix map created successfully: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit/fits/residual.fits


WARNING [CommandLineConfigurator.cc, enableFPExceptions:213]: Floating point exceptions will not be enabled on thisarchitecture
INFO [HealpixSigFluxMap.cc, main:205]: Bin B5C0 maximum padding radius = 2 deg (extension + padding for extended sources)
INFO [HealpixSigFluxMap.cc, main:205]: Bin B6C0 maximum padding radius = 2 deg (extension + padding for extended sources)
INFO [HealpixSigFluxMap.cc, main:205]: Bin B7C0 maximum padding radius = 2 deg (extension + padding for extended sources)
INFO [HealpixSigFluxMap.cc, main:205]: Bin B8C0 maximum padding radius = 2 deg (extension + padding for extended sources)
INFO [HealpixSigFluxMap.cc, main:205]: Bin B9C0 maximum padding radius = 2 deg (extension + padding for extended sources)
INFO [HealpixSigFluxMap.cc, 

In [78]:
from pipeline_helpers import load_hawc_data, find_peak, make_plots
def check_hotspots(path, fit_output, config, logger):
    mapname = path
    l = config.get('coordinates.l')
    b = config.get('coordinates.b')
    x_length = config.get('coordinates.roi_x')
    y_length = config.get('coordinates.roi_y')
    coord_sys = config.get('coordinates.coord_sys')
    array, _, wcs, _, _, pixel_size = load_hawc_data( mapname, l, b, x_length, y_length, coord_sys )
    max_value = find_peak(array, wcs)
    print(f"path.parent: {path.parent.parent}")
    if max_value > 5:
        name = []
        ra = []
        dec = []
        ext = []
        for source in fit_output.model.sources:
            logger.info(f"Source {source}")
            if source == 'URM':
                continue
            try:
                logger.info(f"Source position: RA={fit_output.model[source].position.ra.value}, Dec={fit_output.model[source].position.dec.value}")
                name.append(source)
                ra.append(fit_output.model[source].position.ra.value)
                dec.append(fit_output.model[source].position.dec.value)
                ext.append(0.01)
            except:
                logger.info(f"Source position: RA={fit_output.model[source].spatial_shape.lon0.value}, Dec={fit_output.model[source].spatial_shape.lat0.value}")
                name.append(source)
                ra.append(fit_output.model[source].spatial_shape.lon0.value)
                dec.append(fit_output.model[source].spatial_shape.lat0.value)
                ext.append(fit_output.model[source].spatial_shape.sigma.value)
        df = {'Name': name, 'ra': ra, 'dec': dec, 'ext': ext}
        make_plots(array, wcs, pixel_size, coord_sys, save_dir = str(path.parent), cmap='ult', hotspots=df)
    logger.info(f"Max value in residual map: {max_value}")

In [79]:
check_hotspots(resmap, fit_output, config, logger)

ROI center in Galactic Coordintes = 12.5, 0
Loading Galactic Map


2026-08-20 20:28:20 - Pipeline - INFO - Max value in residual map: 4.942928528398667


Fits File loaded
Degrees per pixel: 0.005555555555555556 
Peak intensity pixel location: (np.int64(841), np.int64(943))
Peak intensity sky location: <SkyCoord (Galactic): (l, b) in deg
    (10.25549306, -0.32220719)>
Peak intensity value: 4.942928528398667
path.parent: /Users/rishi/Documents/Analysis/Sources/AstroImageDetection-fitmodel/HAWC-ImageProcessing-3MLFit/hawc_analysis/Run1/Results/Step1-JointFit


In [87]:
from typing import List
def _as_list(value) -> List[str]:
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


In [109]:
config = ConfigManager('config.yaml')
model = fit_output.model
baseline_log_like = fit_output.log_like
source_names = list(model.sources.keys())
alt_models = _as_list(config.get('fitting.alternate_spatial_models'))
coord_range = config.get('fitting.extended_source_coord_range', 1.0)
logger.info(f"Sources in the model: {source_names}")
logger.info(f"Baseline log-likelihood: {baseline_log_like}")
logger.info(f"Alternate spatial models: {alt_models}")

2026-08-20 21:00:50 - Pipeline - INFO - Sources in the model: ['Source0', 'URM', 'Source1', 'Source2']
2026-08-20 21:00:50 - Pipeline - INFO - Baseline log-likelihood: 117511.16031602172
2026-08-20 21:00:50 - Pipeline - INFO - Alternate spatial models: ['Gaussian_on_sphere']


In [ ]:
result = fit_output 

%load_ext autoreload
%autoreload 2
import importlib
import source_fitter
import model_generator
importlib.reload(source_fitter)
importlib.reload(model_generator)
logger.info('Starting extension fit')


if config.get('fitting.run_extension_test', True):
    result_ext = source_fitter.run_extension_test(result, config, logger, directory_manager)

In [225]:
for sourcename in source_names:
    source = model.sources[sourcename]
    print(f"Checking source {source.name} for extension test")
    if source.name == 'URM':
        params = list(source.spatial_shape.parameters.items())
        # params = {k: v.value for k, v in params}
        params[0][1].free = False
        print(f"Source {source.name} spatial shape: {params[0][1].free}")

for sourcename in source_names:
    source = model.sources[sourcename]
    print(f"Checking source {source.name} for extension test")
    if source.name == 'URM':
        params = list(source.spatial_shape.parameters.items())
        # params = {k: v.value for k, v in params}
        params[0][1].free = True
        print(f"Source {source.name} spatial shape: {params[0][1].free}")


Checking source Source0 for extension test
Checking source URM for extension test
Source URM spatial shape: False
Checking source Source1 for extension test
Checking source Source2 for extension test
Checking source Source0 for extension test
Checking source URM for extension test
Source URM spatial shape: True
Checking source Source1 for extension test
Checking source Source2 for extension test


In [ ]:
%load_ext autoreload
%autoreload 2
import importlib
import model_generator
import source_fitter
importlib.reload(source_fitter)
importlib.reload(model_generator)

alt_models = _as_list(config.get('fitting.alternate_spatial_models'))

for source_name in source_names:
    outdir = directory_manager.get_step_results_dir(f'Step2-{source_name}-Extension')
    name = f'Step2-{source_name}-Extension'
    if source_name == 'URM':
        logger.info(f'Skipping extension test for {source_name} (URM source)')
        continue
    other_sources = [n for n in model.sources.keys() if n != source_name]
    best_log_like = baseline_log_like
    best_model = model

    for alt_shape in alt_models:
        trial_model = model_generator.ModelGenerator.swap_spatial_shape(
            model, source_name, alt_shape, coord_range=coord_range, logger=logger,
        )
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False, logger=logger)
        model_generator.ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=True, logger=logger)
        trial_model.save("{1}/{0}_modelFit.yml".format(name, outdir), overwrite=True)
        # logger.info(trial_model)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spatial', free=False)
        # ModelGenerator.set_free(trial_model, other_sources, kind='spectral', free=False)

        # step_name = f'Step2-{source_name}-Extension-{alt_shape}'
        # step_dir = directory_manager.get_step_results_dir(step_name)
        # model_file = ModelGenerator.write_model_from_live(
        #     trial_model, str(directory_manager.get_model_file_path(step_name)), logger=logger,
        # )

In [ ]:
def initialize_model_file(self,model_file_loc: str, source_info_db: pd.DataFrame):
        '''If model not already in existence make a new one with sources specified by dataframe'''
        # print(source_info_db)
        with open(model_file_loc,'w') as model_file:
            for source in source_info_db['source'].array:
                model_file.write(f'{begin_source_marker}\n')
                model_file.write(f'source_name = \'{source}\'\n\n')
                
                morphology_type = source_info_db.loc[source_info_db['source'] == source,'morphology_type'].values[0]
                morphology_params = source_info_db.loc[source_info_db['source'] == source,'morphology_params'].values[0]
                
                spectrum_type = source_info_db.loc[source_info_db['source'] == source,'spectrum_type'].values[0]
                spectrum_params = source_info_db.loc[source_info_db['source'] == source,'spectrum_params'].values[0]
                
                
                #Add source location params if needed
                if('ra' in morphology_params.keys()):
                    model_file.writelines([f'source_pos_1 = {morphology_params['ra'][0]}\n',f'source_pos_2 = {morphology_params['dec'][0]}\n','\n'])
                elif('lon0' in morphology_params.keys()):
                    model_file.writelines([f'source_pos_1 = {morphology_params['lon0'][0]}\n',f'source_pos_2 = {morphology_params['lat0'][0]}\n','\n'])
                else:
                    self.logger.warning(f'No location for source = {source}. If this is not a template source this is an error.')
                    
                model_file.write(f'spectrum = threeML.{spectrum_type}()\n')
                
                if(morphology_type == 'PointSource'):
                    model_file.write(f'{source} = threeML.PointSource(source_name,ra=source_pos_1,dec=source_pos_2, spectral_shape=spectrum)\n')
                elif(morphology_type == 'Hermes'):
                    model_file.write(f'shape = threeML.{morphology_type}(fits_file=\'{morphology_params['fits_file'][0]}\',ihdu= {morphology_params['ihdu'][0]})\n{source} = threeML.ExtendedSource(source_name,spatial_shape=shape,spectral_shape=spectrum)\n')
                else:
                    model_file.write(f'shape = threeML.{morphology_type}()\n{source} = threeML.ExtendedSource(source_name,spatial_shape=shape,spectral_shape=spectrum)\n')
                    
                model_file.write('fluxUnit = 1. / (threeML.u.keV * threeML.u.cm ** 2 * threeML.u.s)\n')
                
                #loop through spectrum params to define values
                for spectrum_param in spectrum_params.keys():
                    unit_mult = ''
                    if(spectrum_param == 'K'):
                        unit_mult = '* fluxUnit'
                        
                    model_file.writelines(['\n',f'spectrum.{spectrum_param} = {spectrum_params[spectrum_param][0]} {unit_mult}\n',f'spectrum.{spectrum_param}.fix = {not spectrum_params[spectrum_param][3]}\n',f'spectrum.{spectrum_param}.bounds = ({spectrum_params[spectrum_param][1]}, {spectrum_params[spectrum_param][2]}) {unit_mult}\n'])
                #loop through morphology params to define values
                for morphology_param in morphology_params.keys():
                    unit_mult = ''
                    if(morphology_param in ['ra','dec','lon0','lat0']):
                        unit_mult = '* threeML.u.degree'
                    if(morphology_param in ['hash','ihdu','fits_file','frame']):
                        continue
                    if(morphology_type == 'PointSource'):
                        model_file.writelines(['\n',f'{source}.position.{morphology_param}.bounds = ({morphology_params[morphology_param][1]}, {morphology_params[morphology_param][2]}) {unit_mult}\n',f'{source}.position.{morphology_param}.free = {morphology_params[morphology_param][3]}\n'])
                    else:
                        model_file.writelines(['\n',f'shape.{morphology_param} = {morphology_params[morphology_param][0]} {unit_mult}\n',f'shape.{morphology_param}.fix = {not morphology_params[morphology_param][3]}\n',f'shape.{morphology_param}.bounds = ({morphology_params[morphology_param][1]}, {morphology_params[morphology_param][2]}) {unit_mult}\n'])
                model_file.write(f'\n{end_source_marker}\n')
            
            model_file.write(f'model = threeML.Model(')
            for source in source_info_db['source'].array:
                if(not source == source_info_db['source'].array[-1]):
                    model_file.write(f'{source}, ')
                else:
                    model_file.write(f'{source})')
            
        
        return
